In [ ]:
# =========================================================
# 1. Preparação do ambiente
# =========================================================
import pandas as pd
import numpy as np

# =========================================================
# 2. Leitura da base
# =========================================================
df = pd.read_csv('vendas_brasil_aula4_kpis.csv')

print('Primeiras linhas:')
display(df.head())

print('Tamanho da base:')
print(df.shape)

print('Tipos das colunas:')
print(df.dtypes)

print('Informações gerais:')
print(df.info())

# Converter data_venda para datetime
df['data_venda'] = pd.to_datetime(df['data_venda'], errors='coerce')

print('Tipos após conversão de data_venda:')
print(df.dtypes)


# =========================================================
# 4. KPIs globais da operação
# =========================================================
receita_total = df['receita'].sum()
lucro_total = df['lucro'].sum()
margem_lucro_global = lucro_total / receita_total
ticket_medio = df['receita'].mean()

print('\nKPIs Globais')
print(f'Receita Total: R$ {receita_total:,.2f}')
print(f'Lucro Total: R$ {lucro_total:,.2f}')
print(f'Margem de Lucro Global: {margem_lucro_global:.2%}')
print(f'Ticket Médio: R$ {ticket_medio:,.2f}')


# =========================================================
# 5. Métricas derivadas
# =========================================================
df['margem_lucro'] = np.where(df['receita'] != 0, df['lucro'] / df['receita'], np.nan)
df['mes'] = df['data_venda'].dt.month
df['ano'] = df['data_venda'].dt.year
df['ano_mes'] = df['data_venda'].dt.to_period('M').astype(str)
df['trimestre'] = df['data_venda'].dt.quarter

print('\nBase com métricas derivadas:')
display(df.head())


# =========================================================
# 7. Análise por canal
# =========================================================
canal = df.groupby('canal_venda').agg(
    receita_total=('receita', 'sum'),
    lucro_total=('lucro', 'sum'),
    ticket_medio=('receita', 'mean'),
    qtd_pedidos=('pedido_id', 'nunique')
)

canal['margem_lucro'] = canal['lucro_total'] / canal['receita_total']
canal = canal.sort_values(by='receita_total', ascending=False)

print('\nAnálise por canal:')
display(canal)


# =========================================================
# 8. Análise por categoria
# =========================================================
categoria = df.groupby('categoria').agg(
    receita_total=('receita', 'sum'),
    lucro_total=('lucro', 'sum'),
    ticket_medio=('receita', 'mean'),
    qtd_pedidos=('pedido_id', 'nunique')
)

categoria['margem_lucro'] = categoria['lucro_total'] / categoria['receita_total']
categoria = categoria.sort_values(by='lucro_total', ascending=False)

print('\nAnálise por categoria:')
display(categoria)


# =========================================================
# 9. Análise por região (UF)
# =========================================================
uf = df.groupby('uf').agg(
    receita_total=('receita', 'sum'),
    lucro_total=('lucro', 'sum')
)

uf['participacao_receita_%'] = (uf['receita_total'] / uf['receita_total'].sum()) * 100
uf['margem_lucro'] = uf['lucro_total'] / uf['receita_total']
uf = uf.sort_values(by='receita_total', ascending=False)

print('\nAnálise por UF:')
display(uf)


# =========================================================
# 10. Groupby com agregações diferentes
# =========================================================
# Exemplo 1: por canal
groupby_canal = df.groupby('canal_venda').agg(
    receita_total=('receita', 'sum'),
    lucro_total=('lucro', 'sum'),
    media_receita=('receita', 'mean'),
    total_pedidos=('pedido_id', 'count')
).sort_values(by='receita_total', ascending=False)

print('\nGroupby por canal:')
display(groupby_canal)

# Exemplo 2: por UF
groupby_uf = df.groupby('uf').agg(
    receita_total=('receita', 'sum'),
    lucro_total=('lucro', 'sum'),
    media_margem=('margem_lucro', 'mean'),
    total_produtos=('produto', 'count')
).sort_values(by='receita_total', ascending=False)

print('\nGroupby por UF:')
display(groupby_uf)


# =========================================================
# 11. Campeões e vilões (Top N)
# =========================================================
produto_resumo = df.groupby('produto').agg(
    receita_total=('receita', 'sum'),
    lucro_total=('lucro', 'sum')
)

produto_resumo['margem_lucro'] = produto_resumo['lucro_total'] / produto_resumo['receita_total']

top10_lucro = produto_resumo.sort_values(by='lucro_total', ascending=False).head(10)
top10_receita = produto_resumo.sort_values(by='receita_total', ascending=False).head(10)
piores_categorias = categoria.sort_values(by='margem_lucro', ascending=True).head(5)

print('\nTop 10 produtos por lucro:')
display(top10_lucro)

print('\nTop 10 produtos por receita:')
display(top10_receita)

print('\n5 categorias com pior margem:')
display(piores_categorias)


# =========================================================
# 12. Análise temporal
# =========================================================
mensal = df.groupby('ano_mes').agg(
    receita_total=('receita', 'sum'),
    lucro_total=('lucro', 'sum')
)

mensal['margem_lucro'] = mensal['lucro_total'] / mensal['receita_total']
mensal = mensal.sort_index()

print('\nAnálise temporal mensal:')
display(mensal)


# =========================================================
# 16. Desafio extra
# =========================================================
tabela_executiva = pd.DataFrame({
    'Receita Total': [receita_total],
    'Lucro Total': [lucro_total],
    'Margem': [margem_lucro_global],
    'Ticket Médio': [ticket_medio]
})

print('\nTabela executiva final:')
display(tabela_executiva)

canal_executivo = df.groupby('canal_venda').agg(
    receita_total=('receita', 'sum'),
    lucro_total=('lucro', 'sum'),
    ticket_medio=('receita', 'mean')
)

canal_executivo['margem'] = canal_executivo['lucro_total'] / canal_executivo['receita_total']
canal_executivo = canal_executivo.sort_values(by='receita_total', ascending=False)

print('\nTabela executiva por canal:')
display(canal_executivo)

Primeiras linhas:


,pedido_id,data_venda,uf,canal_venda,categoria,produto,quantidade_pedidos,preco_unitario,receita,lucro
0,5000,2025-02-14,RJ,Marketplace,Periféricos,Mouse Gamer,2,237.65,475.30,239.08
1,5001,2025-02-17,PR,Online,Informática,Monitor 27,4,1431.79,5727.16,1769.48
2,5002,2025-07-22,RJ,Marketplace,Informática,Monitor 27,5,1446.22,7231.10,2449.85
3,5003,2025-12-08,SP,Marketplace,Periféricos,Teclado Mecânico,7,399.84,2798.88,1356.67
4,5004,2025-01-04,GO,Marketplace,Telefonia,Smartphone X,4,3134.85,12539.40,3656.84


Tamanho da base:
(420, 10)
Tipos das colunas:
pedido_id               int64
data_venda             object
uf                     object
canal_venda            object
categoria              object
produto                object
quantidade_pedidos      int64
preco_unitario        float64
receita               float64
lucro                 float64
dtype: object
Informações gerais:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 420 entries, 0 to 419
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   pedido_id           420 non-null    int64  
 1   data_venda          420 non-null    object 
 2   uf                  420 non-null    object 
 3   canal_venda         420 non-null    object 
 4   categoria           420 non-null    object 
 5   produto             420 non-null    object 
 6   quantidade_pedidos  420 non-null    int64  
 7   preco_unitario      420 non-null    float64
 8   receita             

,pedido_id,data_venda,uf,canal_venda,categoria,produto,quantidade_pedidos,preco_unitario,receita,lucro,margem_lucro,mes,ano,ano_mes,trimestre
0,5000,2025-02-14,RJ,Marketplace,Periféricos,Mouse Gamer,2,237.65,475.30,239.08,0.503009,2,2025,2025-02,1
1,5001,2025-02-17,PR,Online,Informática,Monitor 27,4,1431.79,5727.16,1769.48,0.308963,2,2025,2025-02,1
2,5002,2025-07-22,RJ,Marketplace,Informática,Monitor 27,5,1446.22,7231.10,2449.85,0.338794,7,2025,2025-07,3
3,5003,2025-12-08,SP,Marketplace,Periféricos,Teclado Mecânico,7,399.84,2798.88,1356.67,0.484719,12,2025,2025-12,4
4,5004,2025-01-04,GO,Marketplace,Telefonia,Smartphone X,4,3134.85,12539.40,3656.84,0.291628,1,2025,2025-01,1



Análise por canal:


...

,receita_total,lucro_total,margem_lucro
ano_mes,,,
2025-01,161571.89,46515.14,0.287891
2025-02,108587.03,29998.87,0.276266
2025-03,87902.22,23923.66,0.272162
2025-04,148930.46,43094.00,0.289357
2025-05,106355.17,30334.32,0.285217
2025-06,134685.83,38592.70,0.286539
2025-07,196651.96,57913.44,0.294497
2025-08,160240.62,45881.36,0.286328
2025-09,109291.03,31938.44,0.292233



Tabela executiva final:


,Receita Total,Lucro Total,Margem,Ticket Médio
0,1921492.22,544496.15,0.283372,4574.981476



Tabela executiva por canal:


,receita_total,lucro_total,ticket_medio,margem
canal_venda,,,,
Marketplace,657802.09,185802.57,4665.263050,0.282460
Loja Física,640797.06,174583.16,4481.098322,0.272447
Online,622893.07,184110.42,4580.096103,0.295573
